In [1]:
import sys
sys.path.append("..")  # permite importar de src/

import pandas as pd
from src.features import limpar_dados, construir_preprocessador, colunas_entrada, ALVO

bruto = pd.read_csv("../data/raw/imoveis.csv")
df = limpar_dados(bruto)

removidas = len(bruto) - len(df)
print(f"{bruto.shape} -> {df.shape}")
print(f"Removidas: {removidas} linhas ({removidas / len(bruto):.1%})")

(10692, 13) -> (10292, 11)
Removidas: 400 linhas (3.7%)


In [2]:
df[["area", "hoa", "iptu", "floor", "aluguel"]].describe().T

,count,mean,std,min,25%,50%,75%,max
area,10292.0,142.714341,129.075260,11.0,58.0,94.0,189.25,1000.0
hoa,10292.0,924.523999,1101.459535,0.0,180.0,570.0,1280.00,9500.0
iptu,10292.0,332.376992,541.948609,0.0,40.0,130.0,384.00,5917.0
floor,10292.0,5.075204,5.307417,0.0,1.0,3.0,8.00,46.0
aluguel,10292.0,3935.716770,3358.372937,450.0,1580.0,2720.0,5000.00,18000.0


In [3]:
bruto["floor"].value_counts().sort_index().tail(15)

floor
20     44
21     42
22     24
23     25
24     19
25     25
26     20
27      8
28      6
29      5
32      2
35      1
46      1
51      1
301     1
Name: count, dtype: int64

In [4]:
X = df[colunas_entrada(incluir_taxas=True)]
prep = construir_preprocessador(incluir_taxas=True)
X_transformado = prep.fit_transform(X)
print(X_transformado.shape)

(10292, 16)


In [5]:
df.to_csv("../data/processed/imoveis_limpo.csv", index=False)

## Limpeza e pré-processamento

- **Linhas removidas:** de 10.692 para 10.292 imóveis, ou seja, 400 linhas
  (3,7%). Cerca de 358 eram duplicatas e as demais, outliers.
- **Outliers (erros de cadastro do scraping):** cortei os valores acima do
  percentil 99,9 em `area`, `hoa`, `iptu` e aluguel, e removi andares acima de
  50. Depois da limpeza, os máximos ficaram plausíveis: área de 1.000 m²
  (antes 46.335), condomínio de R$ 9.500 (antes R$ 1.117.000), IPTU de
  R$ 5.917 (antes R$ 313.700) e aluguel de R$ 18.000. Na análise dos andares,
  os valores 51 e 301 eram isolados e foram removidos, enquanto 35 e 46 ficam
  como possíveis prédios altos reais.
- **Vazamento de dados:** `total` e `fire_insurance` foram descartadas por
  serem derivadas do aluguel. As 231 linhas com aluguel de exatamente R$ 15.000
  foram mantidas (são dados válidos), e o possível teto fica registrado como
  limitação.
- **Pipeline de pré-processamento:** numéricas padronizadas
  (`StandardScaler`) e categóricas com `OneHotEncoder`. Com `hoa` e `iptu` o
  modelo recebe 16 colunas, e sem elas, 14. Na modelagem, tudo ficará dentro de
  um `Pipeline` do scikit-learn, ajustado só nos dados de treino, para não
  haver vazamento entre treino e teste.
- **Dado limpo** salvo em `data/processed/imoveis_limpo.csv`.